# 01 — Exploratory Data Analysis
NSL-KDD network intrusion detection dataset.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TRAIN_CSV, ARTIFACTS_DIR
from src.visualize import plot_class_distribution, plot_correlation_heatmap
from src.preprocessing import CATEGORICAL_COLS

## Load data

In [2]:
df = pd.read_csv(TRAIN_CSV)
print(f'Shape: {df.shape}')
df.head()

Shape: (125973, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## Class distribution

In [3]:
label_counts = df['label'].value_counts().to_dict()
print('Label counts:')
for k, v in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {k:<30} {v:>7,}')
plot_class_distribution(label_counts, title='Label Distribution (Train)', filename='eda_label_dist.png')

Label counts:
  normal                          67,343
  neptune                         41,214
  satan                            3,633
  ipsweep                          3,599
  portsweep                        2,931
  smurf                            2,646
  nmap                             1,493
  back                               956
  teardrop                           892
  warezclient                        890
  pod                                201
  guess_passwd                        53
  buffer_overflow                     30
  warezmaster                         20
  land                                18
  imap                                11
  rootkit                             10
  loadmodule                           9
  ftp_write                            8
  multihop                             7
  phf                                  4
  perl                                 3
  spy                                  2
Saved: /home/abzy/dev/aitu/masters/trimeste

PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/eda_label_dist.png')

## Statistical summary

In [4]:
numeric_cols = [c for c in df.columns if c not in CATEGORICAL_COLS + ['label', 'difficulty']]
df[numeric_cols].describe().T.sort_values('std', ascending=False).head(20)

,count,mean,std,min,25%,50%,75%,max
src_bytes,125973.0,45566.743000,5.870331e+06,0.0,0.00,44.00,276.0,1.379964e+09
dst_bytes,125973.0,19779.114421,4.021269e+06,0.0,0.00,0.00,516.0,1.309937e+09
duration,125973.0,287.144650,2.604515e+03,0.0,0.00,0.00,0.0,4.290800e+04
count,125973.0,84.107555,1.145086e+02,0.0,2.00,14.00,143.0,5.110000e+02
dst_host_srv_count,125973.0,115.653005,1.107027e+02,0.0,10.00,63.00,255.0,2.550000e+02
dst_host_count,125973.0,182.148945,9.920621e+01,0.0,82.00,255.00,255.0,2.550000e+02
srv_count,125973.0,27.737888,7.263584e+01,0.0,2.00,8.00,18.0,5.110000e+02
num_root,125973.0,0.302192,2.439962e+01,0.0,0.00,0.00,0.0,7.468000e+03
num_compromised,125973.0,0.279250,2.394204e+01,0.0,0.00,0.00,0.0,7.479000e+03
hot,125973.0,0.204409,2.149968e+00,0.0,0.00,0.00,0.0,7.700000e+01


## Zero-variance and highly skewed features

In [5]:
variances = df[numeric_cols].var()
zero_var = variances[variances == 0].index.tolist()
print(f'Zero-variance features: {zero_var}')

skew = df[numeric_cols].skew().sort_values(ascending=False)
print(f'\nTop-10 skewed features:')
print(skew.head(10))

Zero-variance features: ['num_outbound_cmds']

Top-10 skewed features:
is_host_login         354.926753
dst_bytes             290.052911
num_compromised       250.107883
num_root              236.913724
src_bytes             190.669347
urgent                149.914509
land                   70.965063
num_shells             59.592151
num_file_creations     55.665341
num_failed_logins      53.764424
dtype: float64


## Feature correlation heatmap

In [6]:
corr = df[numeric_cols].corr().values
plot_correlation_heatmap(corr, numeric_cols, filename='eda_correlation_heatmap.png')

Saved: /home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/eda_correlation_heatmap.png


PosixPath('/home/abzy/dev/aitu/masters/trimester3/aitu-multi-agent-systems/notebooks/../artifacts/eda_correlation_heatmap.png')

## Categorical feature value counts

In [7]:
for col in CATEGORICAL_COLS:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().to_string())


--- protocol_type ---
protocol_type
tcp     102689
udp      14993
icmp      8291

--- service ---
service
http           40338
private        21853
domain_u        9043
smtp            7313
ftp_data        6860
eco_i           4586
other           4359
ecr_i           3077
telnet          2353
finger          1767
ftp             1754
auth             955
Z39_50           862
uucp             780
courier          734
bgp              710
whois            693
uucp_path        689
iso_tsap         687
time             654
imap4            647
nnsp             630
vmnet            617
urp_i            602
domain           569
ctf              563
csnet_ns         545
supdup           544
discard          538
http_443         530
daytime          521
gopher           518
efs              485
systat           477
link             475
exec             474
hostnames        460
name             451
mtp              439
echo             434
klogin           433
login            429
ldap       